# LexIA — Phase 1 : Data Ingestion

Ce notebook retrace la phase d'ingestion du corpus juridique LexIA.
Il couvre :
- La source des données (dump open data DILA)
- Le parsing XML des articles LEGI
- Le filtrage par état juridique
- L'analyse de la distribution du corpus
- Les décisions techniques clés

**Corpus final** : 13 644 articles juridiques en vigueur au 13 juillet 2025  
**Codes** : Code du travail + Code de la consommation  
**Source** : [Freemium LEGI — data.gouv.fr](https://www.data.gouv.fr/datasets/legi-codes-lois-et-reglements-consolides)

## 1. Pourquoi le dump open data plutôt que l'API ?

Légifrance propose une API via le portail PISTE (`api.piste.gouv.fr`).  
Après tests, l'API gratuite retourne **maximum 3 résultats par requête**, quelle que soit la pagination demandée.  
Avec 37 thèmes × 3 résultats = 90 articles — insuffisant pour un RAG sérieux.

**Décision** : utiliser le dump Freemium LEGI (1.1 Go, ~47 000 fichiers XML) qui donne accès à l'intégralité des articles sans limitation.

```bash
# Téléchargement du dump
curl -O "https://echanges.dila.gouv.fr/OPENDATA/LEGI/Freemium_legi_global_20250713-140000.tar.gz"

# Extraction des deux codes uniquement
tar -xzf Freemium_legi_global_20250713-140000.tar.gz \
  --wildcards \
  "legi/global/code_et_TNC_en_vigueur/code_en_vigueur/LEGI/TEXT/*LEGITEXT000006072050*" \
  "legi/global/code_et_TNC_en_vigueur/code_en_vigueur/LEGI/TEXT/*LEGITEXT000006069565*" \
  -C data/raw/legi/
```

## 2. Structure XML du dump LEGI

Chaque article est un fichier `LEGIARTI*.xml` avec la structure suivante :

```xml
<ARTICLE>
  <META>
    <META_COMMUN><ID>LEGIARTI000020690617</ID></META_COMMUN>
    <META_SPEC>
      <META_ARTICLE>
        <NUM>D6325-26</NUM>
        <ETAT>VIGUEUR</ETAT>          <!-- état juridique -->
        <DATE_DEBUT>2009-06-05</DATE_DEBUT>
        <DATE_FIN>2999-01-01</DATE_FIN>
      </META_ARTICLE>
    </META_SPEC>
  </META>
  <CONTEXTE>
    <TEXTE>                           <!-- hiérarchie de sections -->
      <TM><TITRE_TM>Partie réglementaire</TITRE_TM>
        <TM><TITRE_TM>Livre III</TITRE_TM>...</TM>
      </TM>
    </TEXTE>
  </CONTEXTE>
  <BLOC_TEXTUEL>
    <CONTENU><p>Texte de l'article...</p></CONTENU>  <!-- contenu dans des <p> -->
  </BLOC_TEXTUEL>
</ARTICLE>
```

**Point technique clé** : le contenu est enveloppé dans des balises `<p>`, `<br/>`, `<i>` etc.  
`findtext()` retourne `'\n'` — il faut utiliser `itertext()` pour extraire le texte récursivement.

## 3. États juridiques — stratégie de filtrage

Le dump contient **toutes les versions historiques** de chaque article.  
Un même article peut avoir 5-10 versions avec des états différents.

| État | Signification | Conservé ? |
|---|---|---|
| `VIGUEUR` | Article actuellement en vigueur | ✅ |
| `VIGUEUR_DIFF` | Entrera en vigueur prochainement | ✅ |
| `MODIFIE` | Ancienne version remplacée | ❌ |
| `ABROGE` | Abrogé explicitement | ❌ |
| `PERIME` | Périmé | ❌ |
| `ANNULE` | Annulé par le Conseil d'État | ❌ |
| `TRANSFERE` | Transféré dans un autre code | ❌ |
| `MODIFIE_MORT_NE` | Modifié avant son entrée en vigueur | ❌ |

**Logique d'exclusion** plutôt que d'inclusion — on exclut les états invalides, on garde tout le reste.

## 4. Code du parser XML

In [ ]:
import re
import json
from pathlib import Path
from xml.etree import ElementTree as ET
from langchain_core.documents import Document
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

print('Imports OK')

In [ ]:
LEGI_BASE = Path('/workspaces/lexia/data/raw/legi')

CODES = {
    'LEGITEXT000006072050': 'Code du travail',
    'LEGITEXT000006069565': 'Code de la consommation',
}

ETATS_EXCLUS = {
    'MODIFIE', 'ABROGE', 'ABROGE_DIFF',
    'PERIME', 'ANNULE', 'TRANSFERE', 'MODIFIE_MORT_NE',
}

print('Configuration OK')
print(f'Codes ciblés : {list(CODES.values())}')

In [ ]:
def extract_section_hierarchy(contexte):
    """Extrait la hiérarchie de sections depuis le CONTEXTE XML."""
    if contexte is None:
        return ''
    titles = []
    for tm in contexte.iter('TITRE_TM'):
        text = tm.text
        if text and text.strip():
            titles.append(text.strip())
    return ' > '.join(titles)


def extract_contenu(root):
    """
    Extrait le texte via itertext() — nécessaire car le contenu
    est enveloppé dans des balises <p>, <br/>, <i> etc.
    findtext() ne retourne que '\n' sans cette approche.
    """
    contenu_node = root.find('BLOC_TEXTUEL/CONTENU')
    if contenu_node is None:
        return ''
    contenu = ' '.join(contenu_node.itertext())
    contenu = re.sub(r'\s+', ' ', contenu).strip()
    return contenu


def parse_article(xml_path, code_name):
    """Parse un fichier XML LEGIARTI → Document LangChain ou None."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        etat = root.findtext('META/META_SPEC/META_ARTICLE/ETAT', '').strip()
        if etat in ETATS_EXCLUS:
            return None

        contenu = extract_contenu(root)
        if not contenu or len(contenu) < 20:
            return None

        article_id  = root.findtext('META/META_COMMUN/ID', '').strip()
        article_num = root.findtext('META/META_SPEC/META_ARTICLE/NUM', '').strip()
        date_debut  = root.findtext('META/META_SPEC/META_ARTICLE/DATE_DEBUT', '').strip()
        date_fin    = root.findtext('META/META_SPEC/META_ARTICLE/DATE_FIN', '').strip()
        contexte    = root.find('CONTEXTE')
        section     = extract_section_hierarchy(contexte)

        return Document(
            page_content=contenu,
            metadata={
                'source':       'legifrance_dump',
                'code_name':    code_name,
                'article_id':   article_id,
                'article_num':  article_num,
                'section':      section,
                'date_debut':   date_debut,
                'date_fin':     date_fin,
                'etat':         etat,
                'url': f'https://www.legifrance.gouv.fr/codes/article_lc/{article_id}',
            },
        )
    except ET.ParseError:
        return None

print('Fonctions parser définies')

## 5. Chargement du corpus depuis le cache JSONL

Le corpus a déjà été parsé et sauvegardé en `.jsonl`.  
On recharge depuis le cache pour éviter de relancer le parsing (15s par code).

In [ ]:
CORPUS_PATH = '/workspaces/lexia/data/processed/corpus.jsonl'

docs = []
with open(CORPUS_PATH, encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        docs.append(Document(
            page_content=record['page_content'],
            metadata=record['metadata'],
        ))

print(f'Corpus chargé : {len(docs)} documents')
print(f'Codes : {set(d.metadata["code_name"] for d in docs)}')

## 6. Analyse du corpus

In [ ]:
lengths = sorted([len(d.page_content) for d in docs])
n = len(lengths)

print('── Distribution des longueurs ──────────────────')
print(f'Total articles   : {n}')
print(f'Min              : {min(lengths)} chars')
print(f'Max              : {max(lengths)} chars')
print(f'Médiane          : {lengths[n//2]} chars')
print(f'Moyenne          : {sum(lengths)//n} chars')
print(f'P90              : {lengths[int(n*0.90)]} chars')
print(f'P99              : {lengths[int(n*0.99)]} chars')

print()
by_code = Counter(d.metadata['code_name'] for d in docs)
print('── Répartition par code ────────────────────────')
for code, count in by_code.items():
    print(f'  {code:<35} : {count} articles')

print()
by_etat = Counter(d.metadata['etat'] for d in docs)
print('── Répartition par état ────────────────────────')
for etat, count in by_etat.items():
    print(f'  {etat:<20} : {count} articles')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution du corpus LexIA — Phase 1', fontsize=14, fontweight='bold')

# ── Histogramme des longueurs ─────────────────────────────────────────────────
ax1 = axes[0]
lengths_clipped = [min(l, 5000) for l in lengths]
ax1.hist(lengths_clipped, bins=60, color='#1D9E75', edgecolor='white', linewidth=0.5)
ax1.axvline(x=512, color='#D85A30', linestyle='--', linewidth=1.5, label='chunk_size=512')
ax1.axvline(x=lengths[n//2], color='#7F77DD', linestyle='--', linewidth=1.5, label=f'médiane={lengths[n//2]}')
ax1.set_xlabel('Longueur (chars) — tronqué à 5 000')
ax1.set_ylabel('Nombre d\'articles')
ax1.set_title('Distribution des longueurs d\'articles')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# ── Répartition par tranche ───────────────────────────────────────────────────
ax2 = axes[1]
tranches_labels = ['0-200', '200-500', '500-1k', '1k-2k', '2k-5k', '5k+']
tranches_bounds = [(0,200), (200,500), (500,1000), (1000,2000), (2000,5000), (5000,999999)]
counts = [sum(1 for l in lengths if lo <= l < hi) for lo, hi in tranches_bounds]
colors = ['#9FE1CB', '#5DCAA5', '#1D9E75', '#0F6E56', '#085041', '#04342C']

bars = ax2.bar(tranches_labels, counts, color=colors, edgecolor='white', linewidth=0.5)
for bar, count in zip(bars, counts):
    pct = count * 100 // n
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{pct}%', ha='center', va='bottom', fontsize=9)

ax2.set_xlabel('Tranche de longueur')
ax2.set_ylabel('Nombre d\'articles')
ax2.set_title('Articles par tranche de longueur')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/workspaces/lexia/notebooks/corpus_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé → notebooks/corpus_distribution.png')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

codes = list(by_code.keys())
counts = list(by_code.values())
colors_codes = ['#1D9E75', '#7F77DD']

bars = ax.barh(codes, counts, color=colors_codes, edgecolor='white')
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{count:,} articles', va='center', fontsize=10)

ax.set_xlabel('Nombre d\'articles en vigueur')
ax.set_title('Répartition du corpus par code juridique')
ax.set_xlim(0, max(counts) * 1.2)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Inspection d'un article exemple

In [ ]:
# Exemple d'un article moyen (entre 400 et 1000 chars)
sample = next(d for d in docs
              if 400 < len(d.page_content) < 1000
              and d.metadata['code_name'] == 'Code du travail')

print('── Contenu ──────────────────────────────────────────────')
print(sample.page_content)
print()
print('── Métadonnées ──────────────────────────────────────────')
for k, v in sample.metadata.items():
    if k != 'section':
        print(f'  {k:<15} : {v}')
print(f'  {"section":<15} : {sample.metadata["section"][:80]}...')

## 8. Articles outliers (> 10 000 chars)

In [ ]:
outliers = sorted(
    [d for d in docs if len(d.page_content) > 10000],
    key=lambda d: len(d.page_content),
    reverse=True
)

print(f'{len(outliers)} articles > 10 000 chars (traitement spécifique en chunking)\n')
print(f'{"Numéro":<35} {"Longueur":>10}  {"Code"}')
print('-' * 75)
for d in outliers:
    num = d.metadata['article_num'] or 'Annexe'
    print(f'{num:<35} {len(d.page_content):>8} chars  {d.metadata["code_name"]}')

## 9. Conclusions et décisions pour le chunking

### Ce qu'on observe

- **60% des articles font < 500 chars** — ils sont déjà de taille chunk
- **P90 = 1 170 chars** — 90% des articles tiennent en < 1 200 chars
- **16 articles > 10 000 chars** — annexes techniques à traiter différemment
- **Max = 112 726 chars** — annexe technique des règles de sécurité des machines

### Stratégie de chunking (phase 2)

| Catégorie | Seuil | Nb articles | Stratégie |
|---|---|---|---|
| Court | < 400 chars | ~8 270 (60%) | 1 chunk = 1 article |
| Moyen | 400–10 000 chars | ~5 358 (39%) | `chunk_size=512`, `overlap=64` |
| Long (annexes) | > 10 000 chars | 16 (0.1%) | `chunk_size=256`, `overlap=32` |

**Splitter** : `RecursiveCharacterTextSplitter` avec séparateurs `["\n\n", "\n", ".", ";", ",", " "]`  
**Métadonnée parent** : chaque chunk conserve `parent_content` pour le parent-child retrieval

### Limitations du corpus

- Fraîcheur : articles en vigueur au **13 juillet 2025** uniquement
- Périmètre : 2 codes sur 73 disponibles dans le dump
- Extensible : ajouter un code = ajouter son LEGITEXT dans `loader.py`

In [ ]:
# Affiche les métadonnées globales du corpus
with open('/workspaces/lexia/data/processed/corpus_metadata.json') as f:
    meta = json.load(f)

print(json.dumps(meta, indent=2, ensure_ascii=False))